# 1. Import Libraries

In [ ]:
import os
import sys
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from datetime import datetime

# 2. File Paths

In [ ]:
scripts_path = os.path.abspath(os.path.join('..', 'Scripts'))
if scripts_path not in sys.path:
    sys.path.append(scripts_path)

data_dir = os.path.abspath(os.path.join('..', 'Data', 'TrainTest'))
hyperparameter_dir = os.path.abspath(os.path.join('..', 'Data', 'HyperParameters'))
save_dir = os.path.abspath(os.path.join('..', 'Data', 'SavedModels'))
loss_history_dir = os.path.abspath(os.path.join('..', 'Data','LossHistory'))
visualization_dir = os.path.abspath(os.path.join('..', 'Data','Visualization'))

# 3. Load Models & Data

## 1. Data

In [ ]:
train_data = np.load(os.path.join(data_dir, 'train_data.npy')).astype(np.float32)
num_items = train_data.shape[1]

## 2. Models

In [ ]:
from model import Encoder, Decoder, VAE, RSVD

# 4. Training

## 1. VAE

### 1. Import Hyperparameters

In [ ]:
vae_progress_file = os.path.join(hyperparameter_dir, 'tuning_progress_vae.json')
with open(vae_progress_file, 'r') as f:
    best_vae_params = json.load(f)['best_params']

print(f"\n[INFO] Melatih ulang VAE dengan parameter terbaik: {best_vae_params}")

### 2. Construct Model

In [ ]:
encoder = Encoder(hidden_dims=best_vae_params['hidden_dims'], latent_dim=best_vae_params['latent_dim'], dropout_rate=best_vae_params['dropout_rate'])
decoder = Decoder(hidden_dims=best_vae_params['hidden_dims'][::-1], output_dim=num_items)
final_vae = VAE(encoder, decoder)

_ = final_vae(train_data[:1]) 
optimizer = tf.keras.optimizers.Adam(learning_rate=best_vae_params['learning_rate'])
final_vae.compile(optimizer=optimizer)

# Fitur Early Stopping: Berhenti jika loss tidak membaik selama 10 epoch berturut-turut
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='loss', patience=10, restore_best_weights=True, verbose=1
)

### 3. Training Model

In [ ]:
history = final_vae.fit(
    train_data, train_data, 
    epochs=200,
    batch_size=best_vae_params['batch_size'], 
    callbacks=[early_stopping],
    verbose=1 # Tampilkan progress bar agar kita tahu pergerakannya
)

### 4. Save Weights
    Save model weights after finished training

In [ ]:
final_vae.save_weights(os.path.join(save_dir, 'trained_best_vae_weights.weights.h5'))
print("[SUCCESS] Bobot VAE Final berhasil disimpan!")

### 5. Save Loss History
    Save loss history for visualization later

In [ ]:
vae_history_dict = {'loss': history.history['loss']}

with open(os.path.join(loss_history_dir, 'final_vae_loss_history.json'), 'w') as f:
    json.dump(vae_history_dict, f)

### 6. Latent Matrix Z
    Get latent matrix from VAE for training RSVD

In [ ]:
print("\n[INFO] Mengekstrak Matriks Laten Z Final...")
z_mean, _ = final_vae.encoder.predict(train_data, batch_size=best_vae_params['batch_size'])
latent_matrix_Z_final = z_mean

## 2. RSVD

### 1. Import Hyperparameters

In [ ]:
rsvd_progress_file = os.path.join(hyperparameter_dir, 'tuning_progress_rsvd.json')
with open(rsvd_progress_file, 'r') as f:
    best_rsvd_params = json.load(f)['best_params']

print(f"\n[INFO] Melatih ulang RSVD dengan parameter terbaik: {best_rsvd_params}")

### 2. Construct Model

In [ ]:
final_rsvd = RSVD(
    n_factors=best_rsvd_params['n_factors'], 
    learning_rate=best_rsvd_params['learning_rate'], 
    lambda_reg=best_rsvd_params['lambda_reg'], 
    epochs=100 # Naik drastis agar dekomposisi sempurna
)

### 3. Training Model

In [ ]:
final_rsvd.fit(latent_matrix_Z_final)

### 4. Save Weights

In [ ]:
np.save(os.path.join(save_dir, 'best_U.npy'), final_rsvd.U)
np.save(os.path.join(save_dir, 'best_Sigma.npy'), final_rsvd.Sigma)
np.save(os.path.join(save_dir, 'best_V.npy'), final_rsvd.V)
print("[SUCCESS] Matriks RSVD Final berhasil disimpan!")

### 5. Save Loss History
    Save loss history for visualization later

In [ ]:
np.save(os.path.join(loss_history_dir, 'final_rsvd_loss_history.npy'), final_rsvd.loss_history)

# 5. Training Visualization

## 1. Import Loss History

### 1. VAE

In [ ]:
try:
    # Load History Data from Hard Drive
    with open(os.path.join(loss_history_dir, 'final_vae_loss_history.json'), 'r') as f:
        vae_loss_history = json.load(f)['loss']

except FileNotFoundError as e:
    print(f"[ERROR] Could not find the history files. Details: {e}")

### 2. RSVD

In [ ]:
rsvd_loss_history = np.load(os.path.join(loss_history_dir, 'final_rsvd_loss_history.npy'))

## 2. Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=100)
    
# ==========================================
# PLOT 1: VAE LOSS CURVE
# ==========================================
ax1 = axes[0]
ax1.plot(vae_loss_history, label='Total Loss (BCE + KL)', color='#2ca02c', linewidth=2.5)

# Highlight the final epoch
vae_final_epoch = len(vae_loss_history) - 1
vae_final_loss = vae_loss_history[-1]
ax1.scatter(vae_final_epoch, vae_final_loss, color='red', s=50, zorder=5)
ax1.text(vae_final_epoch, vae_final_loss + (vae_final_loss * 0.02), f'{vae_final_loss:.4f}', 
         ha='right', va='bottom', fontsize=12, fontweight='bold', color='red')
         
ax1.set_title('VAE Final Training Convergence', fontsize=15, fontweight='bold', pad=15)
ax1.set_xlabel('Epochs', fontsize=13)
ax1.set_ylabel('Loss Value', fontsize=13)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='upper right', fontsize=11)
# ==========================================
# PLOT 2: RSVD MSE CURVE
# ==========================================
ax2 = axes[1]
ax2.plot(rsvd_loss_history, label='Mean Squared Error (MSE)', color='#1f77b4', linewidth=2.5)

# Highlight the final epoch
rsvd_final_epoch = len(rsvd_loss_history) - 1
rsvd_final_loss = rsvd_loss_history[-1]
ax2.scatter(rsvd_final_epoch, rsvd_final_loss, color='red', s=50, zorder=5)
ax2.text(rsvd_final_epoch, rsvd_final_loss + (rsvd_final_loss * 0.02), f'{rsvd_final_loss:.6f}', 
         ha='right', va='bottom', fontsize=12, fontweight='bold', color='red')
         
ax2.set_title('RSVD Final Training Convergence', fontsize=15, fontweight='bold', pad=15)
ax2.set_xlabel('Epochs', fontsize=13)
ax2.set_ylabel('MSE Value', fontsize=13)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='upper right', fontsize=11)
# ==========================================
# FINALIZE AND DISPLAY
# ==========================================
plt.tight_layout()

# Save the high-resolution image for your thesis report
timestamp = datetime.now().strftime('%Y%m%d_%H%M')
plot_path = os.path.join(visualization_dir, f'final_training_curves_{timestamp}.png')
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"[SUCCESS] High-resolution plot saved at: {plot_path}")

plt.show()